# 🎬 TT_Subtitle — Google Colab Notebook

> Trích xuất phụ đề, dịch thuật và lồng tiếng video từ đa nền tảng.

**Repository gốc:** https://github.com/tpc-pascal/TT_Subtitle  
**Tác giả:** tpc-pascal

---
### 🚀 Hướng dẫn nhanh
1. Chạy từng cell theo thứ tự từ trên xuống dưới.
2. Chọn nền tảng video (YouTube, Bilibili, Douyin...) hoặc dán URL.
3. Nhập từ khóa tìm kiếm hoặc dán link video.
4. Chọn ngôn ngữ gốc và ngôn ngữ dịch.
5. Chọn định dạng đầu ra (SRT, TXT, sub video, lồng tiếng).
6. Chờ xử lý — tải kết quả về máy.

> 💬 Bạn có thể mount Google Drive để lưu file kết quả.
> 💡 Colab có GPU miễn phí, giúp Whisper ASR chạy nhanh hơn.

In [ ]:
# === CELL 1: Cài đặt dependencies ===
print("🔄 Đang cài đặt thư viện...")

!pip install -q gradio yt-dlp openai-whisper transformers sentencepiece protobuf edge-tts moviepy pydub requests httpx
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null 2>&1

print("✅ Hoàn tất cài đặt!")

In [ ]:
# === CELL 2: Import thư viện ===
import os
import sys
import json
import subprocess
import zipfile
import shutil
import warnings
import ipywidgets as widgets
from pathlib import Path
from datetime import datetime
from IPython.display import display, HTML, clear_output, FileLink

warnings.filterwarnings('ignore')

# Clone repo để lấy source utils
if not Path('TT_Subtitle').exists():
    !git clone --depth=1 https://github.com/tpc-pascal/TT_Subtitle.git
sys.path.insert(0, 'TT_Subtitle/hf')

from utils.search import PLATFORMS, SEARCH_PREFIX, search_videos, download_audio, download_video, get_video_info
from utils.subtitle import transcribe, segments_to_srt, segments_to_txt, save_srt, save_txt
from utils.translate import translate_segments
from utils.tts import generate_dub
from utils.video_editor import burn_subtitles, merge_audio_dub, burn_and_dub

OUTPUT_DIR = Path('downloads')
OUTPUT_DIR.mkdir(exist_ok=True)

LANGUAGES = [
    'Tiếng Trung', 'Tiếng Anh', 'Tiếng Việt', 'Tiếng Nhật', 'Tiếng Hàn',
]

print("✅ Import thành công!")

---
### 💾 (Tuỳ chọn) Mount Google Drive
Nếu muốn lưu file kết quả vào Google Drive thay vì tải xuống máy, hãy chạy cell bên dưới.

In [ ]:
# === CELL 3: Mount Google Drive (tuỳ chọn) ===
MOUNT_DRIVE = False  # Đổi thành True nếu muốn mount Drive

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    EXPORT_DIR = Path('/content/drive/MyDrive/TT_Subtitle_Export')
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"📁 Thư mục xuất: {EXPORT_DIR}")
else:
    EXPORT_DIR = OUTPUT_DIR
    print("ℹ️ Dùng thư mục mặc định (tải file xuống máy sau khi xử lý).")

---
### 🎯 Bước 1: Chọn nền tảng & nhập video

In [ ]:
# === CELL 4: Chọn nền tảng ===
platform_dropdown = widgets.Dropdown(
    options=PLATFORMS + ['Khác (dán link)'],
    value='YouTube',
    description='🎯 Nền tảng:',
    style={'description_width': 'initial'},
    layout={'width': '400px'}
)
display(platform_dropown)
PLATFORM = platform_dropdown.value

In [ ]:
# === CELL 5: Tìm kiếm hoặc dán URL ===
PLATFORM = platform_dropdown.value

if PLATFORM in SEARCH_PREFIX:
    keyword = input(f"🔍 Nhập từ khóa tìm kiếm trên {PLATFORM}: ").strip()
    max_results = 10

    if not keyword:
        raise ValueError("❌ Vui lòng nhập từ khóa!")

    print(f"🔄 Đang tìm kiếm {max_results} video trên {PLATFORM}...")
    result = search_videos(PLATFORM, keyword, max_results)

    if not result:
        raise RuntimeError(f"❌ Không tìm thấy video nào cho '{keyword}' trên {PLATFORM}")

    print(f"\n📋 Kết quả tìm kiếm ({len(result)} video):")
    for i, v in enumerate(result, 1):
        dur = v['duration']
        print(f"  {i}. {v['title'][:80]}")
        print(f"     📺 {v['channel']}  ⏱ {dur//60}:{dur%60:02d}  👁 {v.get('view_count', 0):,}")

    choice = input(f"\n👉 Chọn video (1-{len(result)}): ").strip()
    idx = int(choice) - 1
    if idx < 0 or idx >= len(result):
        raise ValueError("❌ Lựa chọn không hợp lệ!")

    VIDEO_URL = result[idx]['url']
    print(f"\n✅ Đã chọn: {result[idx]['title'][:80]}")
    print(f"   🔗 {VIDEO_URL}")

else:
    VIDEO_URL = input("🔗 Dán URL video: ").strip()
    if not VIDEO_URL:
        raise ValueError("❌ Vui lòng nhập URL!")
    print(f"\n✅ URL: {VIDEO_URL}")

---
### 🌐 Bước 2: Cấu hình ngôn ngữ & định dạng đầu ra

In [ ]:
# === CELL 6: Chọn ngôn ngữ và định dạng ===
print("🌐 Chọn ngôn ngữ:")
for i, lang in enumerate(LANGUAGES, 1):
    print(f"  {i}. {lang}")

src_idx = int(input("\n🎯 Ngôn ngữ gốc (nhập số): ").strip())
tgt_idx = int(input("🔄 Ngôn ngữ dịch (nhập số): ").strip())

SOURCE_LANG = LANGUAGES[src_idx - 1]
TARGET_LANG = LANGUAGES[tgt_idx - 1]

print(f"\n📝 Ngôn ngữ: {SOURCE_LANG} → {TARGET_LANG}")

print("\n📦 Định dạng đầu ra:")
formats = ['SRT', 'TXT', 'Sub vào video', 'Cả hai']
for i, fmt in enumerate(formats, 1):
    print(f"  {i}. {fmt}")
fmt_idx = int(input("👉 Chọn (nhập số): ").strip())
OUTPUT_FORMAT = formats[fmt_idx - 1]

dub_input = input("\n🎤 Lồng tiếng bằng AI? (y/N): ").strip().lower()
ENABLE_DUB = dub_input == 'y'

print(f"\n⚙️ Cấu hình:")
print(f"   Định dạng: {OUTPUT_FORMAT}")
print(f"   Lồng tiếng: {'✅ Có' if ENABLE_DUB else '❌ Không'}")

---
### 🚀 Bước 3: Xử lý video

Quá trình này có thể mất vài phút tuỳ vào độ dài video.

In [ ]:
# === CELL 7: Xử lý video ===
print("📥 Bước 1/5: Đang lấy thông tin video...")
info = get_video_info(VIDEO_URL)
print(f"   📺 {info['title']}")
print(f"   📡 {info.get('extractor', 'N/A')}")

print("\n🎵 Bước 2/5: Đang tải âm thanh...")
audio_path = download_audio(VIDEO_URL)
print(f"   ✅ Audio: {audio_path}")

print("\n🤖 Bước 3/5: Đang nhận dạng giọng nói (Whisper ASR)...")
segments = transcribe(audio_path, SOURCE_LANG)
print(f"   ✅ Nhận dạng {len(segments)} đoạn")

if TARGET_LANG != SOURCE_LANG:
    print(f"\n🌐 Bước 3b/5: Đang dịch ({SOURCE_LANG} → {TARGET_LANG})...")
    segments = translate_segments(segments, SOURCE_LANG, TARGET_LANG)
    print(f"   ✅ Dịch xong {len(segments)} đoạn")

sub_srt = segments_to_srt(segments)
sub_txt = segments_to_txt(segments)

print("\n💾 Bước 4/5: Đang xuất file...")
srt_path = save_srt(segments)
txt_path = save_txt(segments)

result_file = srt_path if OUTPUT_FORMAT in ('SRT', 'Cả hai') else txt_path

if OUTPUT_FORMAT in ('Sub vào video', 'Cả hai'):
    print("   Đang tải video gốc...")
    video_path = download_video(VIDEO_URL)
    print("   Đang sub phụ đề vào video...")
    result_file = burn_subtitles(video_path, segments)

if ENABLE_DUB:
    print("\n🎤 Bước 5/5: Đang tạo giọng đọc (TTS)...")
    audio_segs = generate_dub(segments, TARGET_LANG)
    print("   Đang ghép giọng đọc vào video...")

    if OUTPUT_FORMAT in ('Sub vào video', 'Cả hai'):
        video_path = download_video(VIDEO_URL)
        result_file = burn_and_dub(video_path, segments, audio_segs)
    else:
        video_path = download_video(VIDEO_URL)
        result_file = merge_audio_dub(video_path, audio_segs)

print(f"\n\n✅ Hoàn tất xử lý!")
print(f"📁 File kết quả: {result_file}")

---
### 📄 Bước 4: Xem kết quả

In [ ]:
# === CELL 8: Xem phụ đề ===
print(f"📝 Phụ đề ({len(segments)} đoạn):")
for i, seg in enumerate(segments[:20], 1):
    start = f"{int(seg['start']//60)}:{int(seg['start']%60):02d}"
    end = f"{int(seg['end']//60)}:{int(seg['end']%60):02d}"
    print(f"  {i:3d}. [{start} → {end}] {seg['text'][:100]}")

if len(segments) > 20:
    print(f"  ... và {len(segments) - 20} đoạn nữa")

---
### 📥 Bước 5: Tải kết quả

In [ ]:
# === CELL 9: Tải file kết quả ===
if MOUNT_DRIVE:
    dest = EXPORT_DIR / Path(result_file).name
    shutil.copy2(result_file, dest)
    print(f"✅ Đã lưu vào Google Drive: {dest}")
else:
    from google.colab import files
    print("📥 Đang chuẩn bị tải file về máy...")
    files.download(result_file)

# Xuất thêm SRT nếu có
if OUTPUT_FORMAT != 'SRT' and Path(srt_path).exists():
    if not MOUNT_DRIVE:
        from google.colab import files
        files.download(srt_path)

print("\n📊 Tổng kết:")
print(f"   🎬 Video: {info['title'][:80]}")
print(f"   📝 Số đoạn phụ đề: {len(segments)}")
print(f"   🌐 {SOURCE_LANG} → {TARGET_LANG}")
print(f"   📦 Định dạng: {OUTPUT_FORMAT}")
print(f"   🎤 Lồng tiếng: {'✅' if ENABLE_DUB else '❌'}")

In [ ]:
# === CELL 10: Dọn dẹp file tạm (tuỳ chọn) ===
cleanup_input = input("\n🧹 Dọn dẹp file tạm? (y/N): ").strip().lower()

if cleanup_input == 'y':
    from utils.search import cleanup as search_cleanup
    from utils.video_editor import cleanup as editor_cleanup
    search_cleanup()
    editor_cleanup()
    print("✅ Đã dọn dẹp file tạm.")
else:
    print("ℹ️ Giữ lại file tạm.")

---
### ✅ Hoàn tất!

Cảm ơn bạn đã sử dụng **TT_Subtitle**!  
Nếu thấy hữu ích, hãy ghé thăm [GitHub Repository](https://github.com/tpc-pascal/TT_Subtitle) và để lại ⭐.

---
### 📚 Tham khảo thêm

- **Hugging Face Space:** https://huggingface.co/spaces/tpc-pascal/TT_Subtitle  
- **Công nghệ:** [OpenAI Whisper](https://github.com/openai/whisper) · [Meta NLLB](https://ai.meta.com/research/no-language-left-behind/) · [Edge-TTS](https://github.com/rany2/edge-tts) · [yt-dlp](https://github.com/yt-dlp/yt-dlp)